# Sofifa Scraping

In [5]:
import csv
import time
import random
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright
import sys
import asyncio
import os
from playwright_stealth import Stealth

# Force Windows to use the Proactor Event Loop for subprocess support
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION
# ==========================================
# REPLACE THIS with your massive URL containing all 46 columns
BASE_URL = "https://sofifa.com/players?&showCol%5B%5D=pi&showCol%5B%5D=ae&showCol%5B%5D=by&showCol%5B%5D=hi&showCol%5B%5D=pf&showCol%5B%5D=oa&showCol%5B%5D=pt&showCol%5B%5D=bp&showCol%5B%5D=gu&showCol%5B%5D=vl&showCol%5B%5D=wg&showCol%5B%5D=ta&showCol%5B%5D=cr&showCol%5B%5D=fi&showCol%5B%5D=he&showCol%5B%5D=sh&showCol%5B%5D=vo&showCol%5B%5D=ts&showCol%5B%5D=dr&showCol%5B%5D=cu&showCol%5B%5D=fr&showCol%5B%5D=lo&showCol%5B%5D=bl&showCol%5B%5D=to&showCol%5B%5D=ac&showCol%5B%5D=sp&showCol%5B%5D=ag&showCol%5B%5D=tp&showCol%5B%5D=so&showCol%5B%5D=ju&showCol%5B%5D=st&showCol%5B%5D=sr&showCol%5B%5D=ln&showCol%5B%5D=te&showCol%5B%5D=vi&showCol%5B%5D=pe&showCol%5B%5D=td&showCol%5B%5D=ma&showCol%5B%5D=sa&showCol%5B%5D=sl&showCol%5B%5D=tg&showCol%5B%5D=gd&showCol%5B%5D=gh&showCol%5B%5D=gc&showCol%5B%5D=gp&showCol%5B%5D=gr"
CSV_FILENAME = "data/sofifa/newdata/sofifa_players.csv"
TOTAL_PLAYERS = 400000 
PLAYERS_PER_PAGE = 60

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        # SoFifa's first column (the avatar picture) has no text in the header.
        # We dynamically rename this header to "ID" for our CSV.
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    return headers

def extract_rows_from_html(soup, limit=None):
    player_data = []
    rows = soup.select("table tbody tr")
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            # Bulletproof ad blocker (ads have very few columns)
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                # 1. NAME EXTRACTION (Strictly Isolated)
                if 'col-name' in classes:
                    links = td.find_all("a", href=lambda h: h and "/player/" in h)
                    name_text = ""
                    for a in links:
                        # Find the hyperlink that actually has the text of the name, not the avatar image
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        # Fallback just in case
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                # 2. AVATAR/HIDDEN ID EXTRACTION
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/player/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                # 3. ALL OTHER STATS (Including the real ID column)
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
                    
            player_data.append(row_values)
            
            if limit and len(player_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return player_data

# ==========================================
# PHASE 3: THE TURBO BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    os.makedirs(os.path.dirname(CSV_FILENAME), exist_ok=True)

    with sync_playwright() as p:
        # 1. Setup Persistent Profile Folder
        profile_path = os.path.join(os.getcwd(), "data", "sofifa_profile")
        os.makedirs(profile_path, exist_ok=True)
        
        # 2. Launch actual Chrome with saved cookies
        context = p.chromium.launch_persistent_context(
            user_data_dir=profile_path,
            channel="chrome", 
            headless=False,
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        
        # 3. Resource Blocker
        def block_heavy_resources(route):
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]:
                route.abort()
            else:
                route.continue_()
                
        context.route("**/*", block_heavy_resources)
        print("Turbo mode activated (Images/CSS blocked for extreme speed).\n")

        # 4. Grab the default tab and apply Stealth
        page = context.pages[0]
        stealth = Stealth()
        stealth.apply_stealth_sync(page)
        
        print("Launching persistent stealth browser to solve Cloudflare challenge...")
        
        try:
            # 3. Now it is safe to navigate
            page.goto(f"{BASE_URL}&offset=0")
            page.wait_for_selector("table tbody tr", timeout=30000)
            
            # Dynamic wait
            page.wait_for_function(
                "() => document.querySelectorAll('table tbody tr td').length > 100",
                timeout=15000
            )
            print("Challenge passed! Table loaded.")
            
        except Exception as e:
            print(f"Failed to bypass Cloudflare. Error: {e}")
            context.close()
            return

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 10-PLAYER VALIDATION TEST ---")
            
            # # Reload page with Turbo Mode active
            # page.goto(f"{BASE_URL}&offset=0")
            # page.wait_for_selector("table tbody tr") 
            
            soup = BeautifulSoup(page.content(), "html.parser")
            columns = extract_headers_from_html(soup)
            players = extract_rows_from_html(soup, limit=10)

            print(f"Successfully Fetched! Extracted {len(columns)} Columns.")
            print("-" * 50)
            for i, player in enumerate(players):
                player_dict = dict(zip(columns, player))
                print(f"Player {i+1}: {player_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            # 1. Check if we have an existing file to resume from
            start_offset = 0
            if os.path.exists(CSV_FILENAME):
                with open(CSV_FILENAME, "r", encoding="utf-8") as f:
                    # Count lines minus 1 for the header
                    existing_rows = sum(1 for line in f) - 1
                    if existing_rows > 0:
                        # Round down to the nearest multiple of 60 to ensure clean pagination
                        start_offset = (existing_rows // PLAYERS_PER_PAGE) * PLAYERS_PER_PAGE

            print(f"--- STARTING PRODUCTION SCRAPE ---")
            if start_offset > 0:
                print(f"[Resume Mode] Found {existing_rows} existing players. Resuming from offset {start_offset}...\n")
            else:
                print("[New Run] No existing data found. Starting from scratch...\n")

                # Initialize fresh CSV file with headers
                page.goto(f"{BASE_URL}&offset=0")
                page.wait_for_selector("table tbody tr")
                soup = BeautifulSoup(page.content(), "html.parser")
                headers = extract_headers_from_html(soup)
                
                with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                    writer = csv.writer(file)
                    writer.writerow(headers)
                
            # 2. Master Loop (Starts at start_offset)
            with tqdm(total=TOTAL_PLAYERS, initial=start_offset, desc="Scraping SoFifa", unit=" players") as pbar:
                for offset in range(start_offset, TOTAL_PLAYERS, PLAYERS_PER_PAGE):
                    url = f"{BASE_URL}&offset={offset}"
                    
                    success = False
                    for attempt in range(3):
                        try:
                            # Only navigate if it's not the first load (or if we are resuming)
                            if offset != 0 or start_offset > 0 or attempt > 0:
                                page.goto(url)
                                page.wait_for_selector("table tbody tr", timeout=30000)
                                
                                # Dynamic wait: Checks if the whole table has populated
                                page.wait_for_function(
                                    "() => document.querySelectorAll('table tbody tr td').length > 100",
                                    timeout=15000
                                )
                                
                            soup = BeautifulSoup(page.content(), "html.parser")
                            players = extract_rows_from_html(soup)
                            
                            # Append strictly to CSV (mode="a")
                            with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                                writer = csv.writer(file)
                                writer.writerows(players)
                            
                            pbar.update(len(players))
                            success = True
                            
                            if len(players) == 0:
                                print("\n[Notice] No more players found. Database exhausted.")
                                context.close()
                                return
                            break 
                            
                        except Exception as e:
                            print(f"\n[Error on offset {offset}]. Attempt {attempt + 1}/3.")
                            print(f"Details: {str(e)}")  # <--- This will tell us exactly what failed
                            time.sleep(5)
                    
                    if not success:
                        print(f"\n[Fatal] Failed to fetch offset {offset} after 3 attempts. Stopping script to prevent data gaps.")
                        break

                    # Polite delay
                    time.sleep(random.uniform(1.5, 3.0))

            print(f"\nScraping complete! Data safely saved to {CSV_FILENAME}")

        # Safely shut down Chromium
        context.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_17536\2740390964.py:15: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_17536\2740390964.py:15: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [6]:
run_test()

Turbo mode activated (Images/CSS blocked for extreme speed).

Launching persistent stealth browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- RUNNING 10-PLAYER VALIDATION TEST ---
Successfully Fetched! Extracted 50 Columns.
--------------------------------------------------
Player 1: {'Unknown': '', 'Name': 'K. Coulibaly CB CDM CM', 'Age': '18', 'Overall rating': '71 +2', 'Potential': '85 +3', 'Team & Contract': 'SV Werder Bremen 2025 ~ 2029', 'ID': '77636', 'Birth year': '2007', 'Height': '191cm 6\'3"', 'foot': 'Left', 'Best position': 'CB', 'Growth': '14', 'Value': '€4.1M', 'Wage': '€11K', 'Total attacking': '248', 'Crossing': '45 +2', 'Finishing': '35 +2', 'Heading accuracy': '65 +3', 'Short passing': '70 +2', 'Volleys': '33', 'Total skill': '238', 'Dribbling': '50 +3', 'Curve': '38 +4', 'FK Accuracy': '25', 'Long passing': '66 +3', 'Ball control': '59 +1', 'Total movement': '322', 'Acceleration': '61', 'Sprint speed': '68', 'Agility': '62', 'Total power': 

In [ ]:
run_production()

Turbo mode activated (Images/CSS blocked for extreme speed).

Launching persistent stealth browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- STARTING PRODUCTION SCRAPE ---
[Resume Mode] Found 975 existing players. Resuming from offset 960...



Scraping SoFifa:   0%|          | 970/400000 [00:22<253:14:27,  2.28s/ players]